### Analyse von Earnings Calls: Hybrides Modell aus Machine Learning und Large Language Models zur Verringerung von Distraction Bias
**Bachelorarbeit an der Universität St.Gallen (HSG)**\
[© 2026 Ruben Cardell](https://polyformproject.org/licenses/noncommercial/1.0.0)

---

# Postprocessing – Versuche 1, 2 und 3

Dieses Notebook wandelt die Roh-Ausgaben des LLM in die Datensätze für das
Machine Learning um:

1. Ausgaben inspizieren
2. JSON-Objekte in eine Spalte je Sentiment-Dimension auftrennen
3. Bewertungen der Frage-Antwort-Paare je Transkript aggregieren (nur Versuch 1)
4. Tatsächliche Kursentwicklung als Zielvariable ergänzen
5. Datensätze der Versuche 1 und 2 zu Versuch 3 zusammenführen

Die Abschnitte 1 bis 4 werden je Versuch einmal ausgeführt; Abschnitt 3 entfällt
bei Versuch 2, da dort bereits eine Bewertung je Transkript vorliegt.

**Versionsbezeichnungen im Code:** `v3` = Versuch 1, `v4` = Versuch 2,
`v5` = Versuch 3.

In [ ]:
# Google Drive einbinden (Ausführung in Google Colab)
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

Mounted at /content/drive


## 1 Output-Inspektion

In [ ]:
# Roh-Ausgaben des LLM laden (hier Versuch 2; für Versuch 1: output_v3.csv)
df = pd.read_csv('/content/drive/MyDrive/Bachelorarbeit/data/output_v4.csv')

In [ ]:
# Erste Zeilen zur Sichtprüfung
df.head(10)

,episode_id,output
0,AAL_2007_Q3,"```json\n{""relevant"": 0.9, ""clear"": 0.8, ""opti..."
1,AAL_2007_Q4,"```json\n{""relevant"": 0.9, ""clear"": 0.8, ""opti..."
2,AAL_2008_Q1,"```json\n{""relevant"": 0.9, ""clear"": 0.8, ""opti..."
3,AAL_2008_Q2,"{""relevant"": 1.0, ""clear"": 0.9, ""optimistic"": ..."
4,AAL_2008_Q3,"```json\n{""relevant"": 0.9, ""clear"": 0.9, ""opti..."
5,AAL_2008_Q4,"```json\n{""relevant"": 0.9, ""clear"": 0.8, ""opti..."
6,AAL_2009_Q1,"```json\n{""relevant"": 0.9, ""clear"": 0.8, ""opti..."
7,AAL_2009_Q2,"```json\n{""relevant"": 0.9, ""clear"": 0.8, ""opti..."
8,AAL_2009_Q3,"```json\n{""relevant"": 0.95, ""clear"": 0.9, ""opt..."
9,AAL_2009_Q4,"```json\n{""relevant"": 0.95, ""clear"": 0.9, ""opt..."


In [ ]:
# Anzahl Zeilen und fehlende Werte prüfen
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32621 entries, 0 to 32620
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   episode_id  32621 non-null  object
 1   output      32621 non-null  object
dtypes: object(2)
memory usage: 509.8+ KB


In [ ]:
# Format einer einzelnen Modellausgabe prüfen
df.iloc[0]["output"]

'```json\n{"relevant": 1.0, "clear": 1.0, "optimistic": 1.0, "specific": 1.0, "cautious": 0.5, "assertive": 0.9, "valence": 1.0, "arousal": 0.7, "dominance": 0.9, "price": 0.1}\n```'

In [ ]:
# Zweite Ausgabe zum Vergleich
df.iloc[1]["output"]

'{"relevant": 0.5, "clear": 0.5, "optimistic": 0.5, "specific": 0.0, "cautious": 0.5, "assertive": 0.5, "valence": 0.5, "arousal": 0.5, "dominance": 0.5, "price": 0.0}'

## 2 JSON zu Spalten konvertieren

In [ ]:
import json
import re

In [ ]:
def parse_output(s):
    """Wandelt die Roh-Ausgabe des LLM in ein Dictionary um.

    Das Modell gibt das JSON-Objekt teilweise in einem Markdown-Codeblock aus;
    dessen Auszeichnung wird zuerst entfernt. Nicht parsbare Ausgaben werden
    protokolliert und als None zurückgegeben, sodass sie später als fehlende
    Werte erscheinen.
    """
    s = re.sub(r'^```json\s*|\s*```$', '', s.strip())
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        print(f"Failed to parse: {s}")
        return None

In [ ]:
# JSON-Objekte parsen und in eine Spalte je Dimension überführen
parsed = df['output'].apply(parse_output)
features = pd.json_normalize(parsed)  # NaN-Zeilen, wo das Parsen fehlgeschlagen ist

df = pd.concat([df.drop(columns='output'), features], axis=1)

In [ ]:
# Erste Zeilen der aufgetrennten Features
df.head()

,episode_id,relevant,clear,optimistic,specific,cautious,assertive,valence,arousal,dominance,price_1d,price_30d,price_90d
0,AAL_2007_Q3,0.9,0.8,0.4,0.8,0.9,0.6,0.2,0.3,0.5,0.02,0.03,0.04
1,AAL_2007_Q4,0.9,0.8,0.3,0.7,0.9,0.5,0.1,0.2,0.6,0.01,-0.03,-0.05
2,AAL_2008_Q1,0.9,0.8,0.3,0.7,0.9,0.6,0.1,0.2,0.5,0.01,-0.03,-0.05
3,AAL_2008_Q2,1.0,0.9,0.7,0.9,0.8,0.8,0.5,0.4,0.7,0.05,0.08,0.10
4,AAL_2008_Q3,0.9,0.9,0.7,0.8,0.7,0.8,0.7,0.5,0.8,0.02,0.05,0.08


In [ ]:
# Verteilung der Feature-Werte prüfen
df.describe()

,relevant,clear,optimistic,specific,cautious,assertive,valence,arousal,dominance,price_1d,price_30d,price_90d
count,32621.000000,32621.000000,32621.000000,32621.000000,32621.00000,32621.000000,32621.000000,32621.000000,32621.000000,32621.000000,32621.000000,32621.000000
mean,0.928184,0.867536,0.718801,0.787059,0.74451,0.748001,0.574434,0.410907,0.767320,0.022188,0.045326,0.067760
std,0.031459,0.048898,0.137061,0.105264,0.07316,0.082408,0.194179,0.120569,0.073668,0.006381,0.021448,0.032241
min,0.500000,0.400000,0.100000,0.000000,0.10000,0.200000,0.000000,0.000000,0.200000,-0.080000,-0.150000,-0.250000
25%,0.900000,0.800000,0.600000,0.700000,0.70000,0.700000,0.500000,0.300000,0.700000,0.020000,0.040000,0.060000
50%,0.950000,0.900000,0.700000,0.800000,0.70000,0.800000,0.600000,0.400000,0.800000,0.020000,0.050000,0.080000
75%,0.950000,0.900000,0.800000,0.850000,0.80000,0.800000,0.700000,0.500000,0.800000,0.030000,0.050000,0.080000
max,1.000000,1.000000,1.000000,1.000000,0.90000,0.950000,1.000000,0.850000,1.000000,0.080000,0.120000,0.180000


In [ ]:
# Qualitätskontrolle: Zeilen, in denen das Modell alle Dimensionen nur mit
# den Randwerten 0.0 oder 1.0 bewertet hat
feature_cols = [c for c in features.columns if c not in ['price_1d', 'price_30d', 'price_90d']]

mask = df[feature_cols].isin([0.0, 1.0]).all(axis=1)
count = mask.sum()

print(f"Rows with only 0.0 or 1.0 in feature columns: {count}")

Rows with only 0.0 or 1.0 in feature columns: 0


## 3 Aggregation der Frage-Antwort-Paare (nur Versuch 1)

In Versuch 1 liegt je Frage-Antwort-Paar eine Bewertung vor. Für das
anschliessende Machine Learning werden diese je Transkript zu Mittelwert,
Minimum und Maximum aggregiert: Der Mittelwert gibt das Niveau über den
gesamten Call wieder, Minimum und Maximum die auffälligste Antwort in jede
Richtung. Aus den neun Dimensionen und der Direktprognose entstehen so 30
Merkmale je Transkript.

In [ ]:
feature_cols = [c for c in features.columns]  # einschliesslich der Direktprognose

grouped_df = df.groupby('episode_id')[feature_cols].agg(['mean', 'min', 'max'])

# Mehrstufige Spaltennamen zusammenführen, z.B. relevant_mean, relevant_min, relevant_max
grouped_df.columns = [f"{col}_{stat}" for col, stat in grouped_df.columns]
grouped_df = grouped_df.reset_index()

In [ ]:
# Ergebnis der Aggregation prüfen
grouped_df

,episode_id,relevant_mean,relevant_min,relevant_max,clear_mean,clear_min,clear_max,optimistic_mean,optimistic_min,optimistic_max,...,valence_max,arousal_mean,arousal_min,arousal_max,dominance_mean,dominance_min,dominance_max,price_mean,price_min,price_max
0,AAL_2007_Q3,0.950000,0.8,1.0,0.830000,0.6,1.0,0.300000,0.1,0.7,...,0.5,0.250000,0.2,0.4,0.550000,0.3,0.8,0.005000,-0.03,0.03
1,AAL_2007_Q4,0.910000,0.7,1.0,0.760000,0.3,1.0,0.250000,0.1,0.6,...,0.4,0.200000,0.1,0.3,0.480000,0.3,0.7,-0.004000,-0.03,0.03
2,AAL_2008_Q1,0.960000,0.8,1.0,0.710000,0.5,0.8,0.220000,0.1,0.5,...,0.3,0.230000,0.2,0.3,0.440000,0.3,0.6,-0.005000,-0.03,0.02
3,AAL_2008_Q2,0.931250,0.3,1.0,0.706250,0.0,1.0,0.381250,0.1,0.7,...,0.8,0.243750,0.1,0.4,0.606250,0.4,0.8,0.013750,-0.03,0.03
4,AAL_2008_Q3,0.925000,0.7,1.0,0.668750,0.0,1.0,0.462500,0.1,0.8,...,1.0,0.268750,0.1,0.4,0.568750,0.3,0.8,0.013125,-0.03,0.03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32616,ZTS_2024_Q1,0.890909,0.5,1.0,0.645455,0.0,1.0,0.454545,0.0,0.9,...,0.9,0.318182,0.1,0.6,0.572727,0.1,0.8,0.038182,0.00,0.08
32617,ZTS_2024_Q2,0.990909,0.9,1.0,0.872727,0.7,1.0,0.809091,0.6,1.0,...,1.0,0.481818,0.3,0.7,0.754545,0.5,0.9,0.065455,0.03,0.10
32618,ZTS_2024_Q3,0.990000,0.9,1.0,0.820000,0.7,0.9,0.740000,0.2,0.9,...,0.9,0.420000,0.2,0.6,0.740000,0.6,0.9,0.062000,0.03,0.10
32619,ZTS_2024_Q4,0.936364,0.3,1.0,0.754545,0.0,1.0,0.781818,0.1,1.0,...,1.0,0.490909,0.1,0.7,0.718182,0.1,0.9,0.070000,0.00,0.10


In [ ]:
# Verteilung der aggregierten Merkmale prüfen
print(grouped_df.describe())

       relevant_mean  relevant_min  relevant_max    clear_mean     clear_min  \
count   32621.000000  32621.000000  32621.000000  32621.000000  32621.000000   
mean        0.965138      0.805795      0.999094      0.794112      0.351614   
std         0.038662      0.179190      0.017307      0.103416      0.345471   
min         0.500000      0.000000      0.500000      0.000000      0.000000   
25%         0.950000      0.700000      1.000000      0.733333      0.000000   
50%         0.975000      0.800000      1.000000      0.800000      0.300000   
75%         0.991667      0.900000      1.000000      0.870000      0.700000   
max         1.000000      1.000000      1.000000      1.000000      1.000000   

          clear_max  optimistic_mean  optimistic_min  optimistic_max  \
count  32621.000000     32621.000000    32621.000000    32621.000000   
mean       0.987873         0.581925        0.263179        0.827225   
std        0.048172         0.121109        0.182398        0.1

## 4 Tatsächliche Kursentwicklung ergänzen

Die Zielvariable `move_next_qtr` ist die Kursentwicklung über die 90 Tage nach
dem Earnings Call. `move_1d` und `move_30d` werden mitgeführt, in den Versuchen
aber nicht verwendet.

In [ ]:
# Tatsächliche Kursentwicklungen je Transkript laden
price_df = pd.read_parquet('/content/drive/MyDrive/Bachelorarbeit/data/features.parquet', columns = ["episode_id", "move_1d", "move_30d", "move_next_qtr"])

In [ ]:
# Kursdaten zur Sichtprüfung
price_df

,episode_id,move_1d,move_30d,move_next_qtr
0,A_2006_Q1,-0.029830,0.046278,0.038472
1,A_2006_Q2,-0.064429,-0.143087,-0.269799
2,A_2006_Q3,0.144117,0.152573,0.365675
3,A_2006_Q4,-0.053143,-0.026857,-0.058857
4,A_2007_Q1,-0.005161,-0.033091,0.099272
...,...,...,...,...
33357,ZTS_2024_Q1,0.054069,0.087508,0.105589
33358,ZTS_2024_Q2,0.056916,0.067326,0.043251
33359,ZTS_2024_Q3,-0.036713,-0.036438,-0.041464
33360,ZTS_2024_Q4,-0.094088,-0.050092,-0.087915


In [ ]:
# Kursdaten je Transkript anfügen (für Versuch 1: grouped_df statt df)
ml_df = pd.merge(df, price_df, how="left", on="episode_id")

In [ ]:
# Vollständigkeit des fertigen ML-Datensatzes prüfen
ml_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32621 entries, 0 to 32620
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   episode_id     32621 non-null  object 
 1   relevant       32621 non-null  float64
 2   clear          32621 non-null  float64
 3   optimistic     32621 non-null  float64
 4   specific       32621 non-null  float64
 5   cautious       32621 non-null  float64
 6   assertive      32621 non-null  float64
 7   valence        32621 non-null  float64
 8   arousal        32621 non-null  float64
 9   dominance      32621 non-null  float64
 10  price_1d       32621 non-null  float64
 11  price_30d      32621 non-null  float64
 12  price_90d      32621 non-null  float64
 13  move_1d        29514 non-null  float64
 14  move_30d       29514 non-null  float64
 15  move_next_qtr  28934 non-null  float64
dtypes: float64(15), object(1)
memory usage: 4.0+ MB


In [ ]:
# Fertigen Datensatz für das Modelltraining speichern
ml_df.to_csv('/content/drive/MyDrive/Bachelorarbeit/data/ml_data_v4.csv', index=False)

## 5 Datensatz für Versuch 3 zusammensetzen

Versuch 3 beinhaltet keinen neuen LLM-Lauf, sondern führt die Datensätze der
Versuche 1 und 2 zusammen: 27 aggregierte Merkmale aus den Frage-Antwort-Paaren
und 9 Merkmale aus den gesamten Transkripten. Die Direktprognosen des LLM gehen
nicht als Features ein, sondern dienen ausschliesslich als Vergleichsgrösse.

In [ ]:
# Versuch 1: Direktprognosen und ungenutzte Kursspalten entfernen
df_v3 = pd.read_csv('/content/drive/MyDrive/Bachelorarbeit/data/ml_data_v3.csv', index_col="episode_id").drop(columns=["price_mean", "price_min", "price_max", "move_1d", "move_30d"])

In [ ]:
# Versuch 2: zusätzlich die Zielvariable entfernen, sie kommt aus df_v3
df_v4 = pd.read_csv('/content/drive/MyDrive/Bachelorarbeit/data/ml_data_v4.csv', index_col="episode_id").drop(columns=["price_1d", "price_30d", "price_90d", "move_1d", "move_30d", "move_next_qtr"])

In [ ]:
# Beide Merkmalssätze über die episode_id verbinden
df_v5 = df_v3.join(df_v4)

In [ ]:
# Ergebnis prüfen: 36 Features plus Zielvariable
df_v5

,relevant_mean,relevant_min,relevant_max,clear_mean,clear_min,clear_max,optimistic_mean,optimistic_min,optimistic_max,specific_mean,...,move_next_qtr,relevant,clear,optimistic,specific,cautious,assertive,valence,arousal,dominance
episode_id,,,,,,,,,,,,,,,,,,,,,
AAL_2007_Q3,0.950000,0.8,1.0,0.830000,0.6,1.0,0.300000,0.1,0.7,0.500000,...,-0.565474,0.90,0.8,0.40,0.80,0.9,0.60,0.20,0.3,0.50
AAL_2007_Q4,0.910000,0.7,1.0,0.760000,0.3,1.0,0.250000,0.1,0.6,0.480000,...,-0.348521,0.90,0.8,0.30,0.70,0.9,0.50,0.10,0.2,0.60
AAL_2008_Q1,0.960000,0.8,1.0,0.710000,0.5,0.8,0.220000,0.1,0.5,0.390000,...,-0.669939,0.90,0.8,0.30,0.70,0.9,0.60,0.10,0.2,0.50
AAL_2008_Q2,0.931250,0.3,1.0,0.706250,0.0,1.0,0.381250,0.1,0.7,0.531250,...,2.156134,1.00,0.9,0.70,0.90,0.8,0.80,0.50,0.4,0.70
AAL_2008_Q3,0.925000,0.7,1.0,0.668750,0.0,1.0,0.462500,0.1,0.8,0.531250,...,-0.140165,0.90,0.9,0.70,0.80,0.7,0.80,0.70,0.5,0.80
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZTS_2024_Q1,0.890909,0.5,1.0,0.645455,0.0,1.0,0.454545,0.0,0.9,0.400000,...,0.105589,0.90,0.8,0.90,0.70,0.6,0.80,0.80,0.5,0.80
ZTS_2024_Q2,0.990909,0.9,1.0,0.872727,0.7,1.0,0.809091,0.6,1.0,0.790909,...,0.043251,0.95,0.9,0.95,0.90,0.7,0.85,0.90,0.6,0.80
ZTS_2024_Q3,0.990000,0.9,1.0,0.820000,0.7,0.9,0.740000,0.2,0.9,0.580000,...,-0.041464,0.95,0.9,0.95,0.85,0.7,0.85,0.90,0.6,0.80


In [ ]:
# Datensatz für Versuch 3 speichern
df_v5.to_csv('/content/drive/MyDrive/Bachelorarbeit/data/ml_data_v5.csv', index=True)